# Electric forecast historical data scraping

## Import libraries

In [6]:
import os
import time
import requests
import zipfile
from io import StringIO, BytesIO
from bs4 import BeautifulSoup
from datetime import date,datetime
import polars as pl
import numpy as np

## Scrape historical data

### Tokyo

In [2]:
BASE_URL = "https://www.tepco.co.jp/forecast/html/images"
OUTPUT_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tokyo"

#### Scrape raw csv files

In [4]:
for year in range(2016, 2023):
    file_name = f"juyo-{year}.csv"
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)

    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=2)
        df.write_parquet(os.path.join(OUTPUT_PATH, f"juyo-{year}.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

    time.sleep(np.random.uniform(1, 3))

Processing data for year 2016
Saved parquet for year 2016
Processing data for year 2017
Saved parquet for year 2017
Processing data for year 2018
Saved parquet for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022


In [5]:
tokyo_parquet = r"/workspace/src/stg/data_lake/electric_forecast/tokyo/juyo-2022.parquet"
df_tokyo_2022 = pl.read_parquet(tokyo_parquet)
df_tokyo_2022.head()

DATE,TIME,実績(万kW)
str,str,i64
"""2022/1/1""","""0:00""",3266
"""2022/1/1""","""1:00""",3062
"""2022/1/1""","""2:00""",2929
"""2022/1/1""","""3:00""",2828
"""2022/1/1""","""4:00""",2786


#### Scrape Zip files

In [ ]:
ZIP_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tokyo/zip"
year_list = [i for i in range(2022,2026)]
month_list = [i for i in range(1,13)]
print(year_list,month_list)

[2022, 2023, 2024, 2025] [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [ ]:
#for year in year_list:
#for month in range(2,13):
zip_file_name = "202512_power_usage.zip"
zip_url = f"{BASE_URL}/{zip_file_name}"
response = requests.get(zip_url)

if response.status_code == 200:
    with open(os.path.join(ZIP_PATH, zip_file_name), 'wb') as f:
        f.write(response.content)
    print(f"Downloaded {zip_file_name}")
else:
    print(f"Failed to download {zip_file_name}")

    time.sleep(np.random.uniform(3, 6))

Downloaded 202502_power_usage.zip
Downloaded 202503_power_usage.zip
Downloaded 202504_power_usage.zip
Downloaded 202505_power_usage.zip
Downloaded 202506_power_usage.zip
Downloaded 202507_power_usage.zip
Downloaded 202508_power_usage.zip
Downloaded 202509_power_usage.zip
Downloaded 202510_power_usage.zip
Downloaded 202511_power_usage.zip
Downloaded 202512_power_usage.zip


### Hokkaido

In [16]:
HKD_PATH = r"/workspace/src/stg/data_lake/electric_forecast/hokkaido"
HKD_BASE_URL = "https://denkiyoho.hepco.co.jp/area/data/zip"
QUARTER = {
    1:"04-06",
    2:"07-09",
    3:"10-12",
    4:"01-03"
}
HKD_YEAR_LIST = [i for i in range(2020,2026)]

In [17]:
for year in HKD_YEAR_LIST:
    for quarter in QUARTER.values():
        zip_file_name = f"{year}{quarter}_hokkaido_denkiyohou.zip"
        zip_url = f"{HKD_BASE_URL}/{zip_file_name}"
        response = requests.get(zip_url)

        if response.status_code == 200:
            with open(os.path.join(HKD_PATH, zip_file_name), 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {zip_file_name}")
        else:
            print(f"Failed to download {zip_file_name}")

            time.sleep(np.random.uniform(1, 5))

Downloaded 202004-06_hokkaido_denkiyohou.zip
Downloaded 202007-09_hokkaido_denkiyohou.zip
Downloaded 202010-12_hokkaido_denkiyohou.zip
Failed to download 202001-03_hokkaido_denkiyohou.zip
Downloaded 202104-06_hokkaido_denkiyohou.zip
Downloaded 202107-09_hokkaido_denkiyohou.zip
Downloaded 202110-12_hokkaido_denkiyohou.zip
Downloaded 202101-03_hokkaido_denkiyohou.zip
Downloaded 202204-06_hokkaido_denkiyohou.zip
Downloaded 202207-09_hokkaido_denkiyohou.zip
Downloaded 202210-12_hokkaido_denkiyohou.zip
Downloaded 202201-03_hokkaido_denkiyohou.zip
Downloaded 202304-06_hokkaido_denkiyohou.zip
Downloaded 202307-09_hokkaido_denkiyohou.zip
Downloaded 202310-12_hokkaido_denkiyohou.zip
Downloaded 202301-03_hokkaido_denkiyohou.zip
Downloaded 202404-06_hokkaido_denkiyohou.zip
Downloaded 202407-09_hokkaido_denkiyohou.zip
Downloaded 202410-12_hokkaido_denkiyohou.zip
Downloaded 202401-03_hokkaido_denkiyohou.zip
Downloaded 202504-06_hokkaido_denkiyohou.zip
Downloaded 202507-09_hokkaido_denkiyohou.zip
Do

### Tohoku

In [19]:
THK_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tohoku"
THK_BASE_URL = "https://setsuden.nw.tohoku-epco.co.jp/common/demand/"
THK_YEAR_LIST = [i for i in range(2016,2026)]

In [20]:
for year in THK_YEAR_LIST:
    file_name = f"juyo_{year}_tohoku.csv"
    url = f"{THK_BASE_URL}/{file_name}"
    response = requests.get(url)

    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=1)
        df.write_parquet(os.path.join(THK_PATH, f"juyo_{year}_tohoku.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

    time.sleep(np.random.uniform(1, 5))

Processing data for year 2016
Saved parquet for year 2016
Processing data for year 2017
Saved parquet for year 2017
Processing data for year 2018
Saved parquet for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022
Processing data for year 2023
Saved parquet for year 2023
Processing data for year 2024
Saved parquet for year 2024
Processing data for year 2025
Saved parquet for year 2025


### Chubu

In [21]:
CHB_PATH = r"/workspace/src/stg/data_lake/electric_forecast/chubu/zip"
CHB_BASE_URL = "https://powergrid.chuden.co.jp/denki_yoho_content_data/download_csv/"
CHB_YEAR_LIST = [i for i in range(2019,2026)]
CHB_MONTH_LIST = [i for i in range(1,13)]

In [24]:
#201912_power_usage.zip
for year in CHB_YEAR_LIST:
    for month in CHB_MONTH_LIST:
        zip_file_name = f"{year}{str(month).zfill(2)}_power_usage.zip"
        zip_url = f"{CHB_BASE_URL}/{zip_file_name}"
        time.sleep(np.random.uniform(1, 5))
        response = requests.get(zip_url)
        if response.status_code == 200:
            with open(os.path.join(CHB_PATH, zip_file_name), 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {zip_file_name}")
        else:
            print(f"Failed to download {zip_file_name}")

Failed to download 201901_power_usage.zip
Failed to download 201902_power_usage.zip
Failed to download 201903_power_usage.zip
Downloaded 201904_power_usage.zip
Downloaded 201905_power_usage.zip
Downloaded 201906_power_usage.zip
Downloaded 201907_power_usage.zip
Downloaded 201908_power_usage.zip
Downloaded 201909_power_usage.zip
Downloaded 201910_power_usage.zip
Downloaded 201911_power_usage.zip
Downloaded 201912_power_usage.zip
Downloaded 202001_power_usage.zip
Downloaded 202002_power_usage.zip
Downloaded 202003_power_usage.zip
Downloaded 202004_power_usage.zip
Downloaded 202005_power_usage.zip
Downloaded 202006_power_usage.zip
Downloaded 202007_power_usage.zip
Downloaded 202008_power_usage.zip
Downloaded 202009_power_usage.zip
Downloaded 202010_power_usage.zip
Downloaded 202011_power_usage.zip
Downloaded 202012_power_usage.zip
Downloaded 202101_power_usage.zip
Downloaded 202102_power_usage.zip
Downloaded 202103_power_usage.zip
Downloaded 202104_power_usage.zip
Downloaded 202105_power_

### Kansai

In [3]:
KNS_PATH = r"/workspace/src/stg/data_lake/electric_forecast/kansai"
KNS_BASE_URL = "https://www.kansai-td.co.jp/yamasou"
KNS_YEAR_LIST = [i for i in range(2016,2026)]
KNS_MONTH_LIST = [i for i in range(1,13)]

In [6]:
for year in KNS_YEAR_LIST:
    for month in KNS_MONTH_LIST:
        zip_file_name = f"{year}{str(month).zfill(2)}_jisseki.zip"
        zip_url = f"{KNS_BASE_URL}/{zip_file_name}"
        time.sleep(np.random.uniform(1, 5))
        response = requests.get(zip_url)
        if response.status_code == 200:
            with open(os.path.join(KNS_PATH, zip_file_name), 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {zip_file_name}")
        else:
            print(f"Failed to download {zip_file_name}")

Failed to download 201601_jisseki.zip
Failed to download 201602_jisseki.zip
Failed to download 201603_jisseki.zip
Downloaded 201604_jisseki.zip
Downloaded 201605_jisseki.zip
Downloaded 201606_jisseki.zip
Downloaded 201607_jisseki.zip
Downloaded 201608_jisseki.zip
Downloaded 201609_jisseki.zip
Downloaded 201610_jisseki.zip
Downloaded 201611_jisseki.zip
Downloaded 201612_jisseki.zip
Downloaded 201701_jisseki.zip
Downloaded 201702_jisseki.zip
Downloaded 201703_jisseki.zip
Downloaded 201704_jisseki.zip
Downloaded 201705_jisseki.zip
Downloaded 201706_jisseki.zip
Downloaded 201707_jisseki.zip
Downloaded 201708_jisseki.zip
Downloaded 201709_jisseki.zip
Downloaded 201710_jisseki.zip
Downloaded 201711_jisseki.zip
Downloaded 201712_jisseki.zip
Downloaded 201801_jisseki.zip
Downloaded 201802_jisseki.zip
Downloaded 201803_jisseki.zip
Downloaded 201804_jisseki.zip
Downloaded 201805_jisseki.zip
Downloaded 201806_jisseki.zip
Downloaded 201807_jisseki.zip
Downloaded 201808_jisseki.zip
Downloaded 20180

### Chugoku

In [10]:
CGK_PATH = r"/workspace/src/stg/data_lake/electric_forecast/chugoku"
CGK_URL = "https://www.energia.co.jp/nw/jukyuu/sys/"
CGK_YEAR_LIST = [i for i in range(2010,2026)]

In [11]:
for year in CGK_YEAR_LIST:
    file_name = f"juyo-{str(year)}.csv"
    url = f"{CGK_URL}/{file_name}"
    time.sleep(np.random.uniform(1, 5))
    response = requests.get(url)
    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=2)
        df.write_parquet(os.path.join(CGK_PATH, f"juyo-{year}.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

Failed to retrieve data for year 2010
Failed to retrieve data for year 2011
Failed to retrieve data for year 2012
Failed to retrieve data for year 2013
Failed to retrieve data for year 2014
Failed to retrieve data for year 2015
Failed to retrieve data for year 2016
Failed to retrieve data for year 2017
Failed to retrieve data for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022
Processing data for year 2023
Saved parquet for year 2023
Processing data for year 2024
Saved parquet for year 2024
Processing data for year 2025
Saved parquet for year 2025


In [15]:
df_chugoku_2021 = pl.read_parquet(r"/workspace/src/stg/data_lake/electric_forecast/chugoku/juyo-2021.parquet")
df_chugoku_2021.head()

DATE,TIME,実績(万kW)
str,str,i64
"""2021/4/1""","""0:00""",520
"""2021/4/1""","""1:00""",529
"""2021/4/1""","""2:00""",566
"""2021/4/1""","""3:00""",609
"""2021/4/1""","""4:00""",622


### Shikoku

In [12]:
SKK_PATH = r"/workspace/src/stg/data_lake/electric_forecast/shikoku"
SKK_URL = "https://www.yonden.co.jp/nw/denkiyoho/csv/"
SKK_YEAR_LIST = [i for i in range(2016,2026)]

In [18]:
for year in SKK_YEAR_LIST:
    file_name = f"juyo_shikoku_{str(year)}.csv"
    url = f"{SKK_URL}/{file_name}"
    time.sleep(np.random.uniform(1, 5))
    response = requests.get(url)
    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=2)
        df.write_parquet(os.path.join(SKK_PATH, f"juyo_shikoku_{str(year)}.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

Processing data for year 2016
Saved parquet for year 2016
Processing data for year 2017
Saved parquet for year 2017
Processing data for year 2018
Saved parquet for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022
Processing data for year 2023
Saved parquet for year 2023
Processing data for year 2024
Saved parquet for year 2024
Processing data for year 2025
Saved parquet for year 2025


### KyuShu

In [2]:
KYS_PATH = r"/workspace/src/stg/data_lake/electric_forecast/kyushu"
KYS_URL = "https://www.kyuden.co.jp/td_power_usages/csv/"
KYS_YEAR_LIST = [i for i in range(2011,2026)]
KYS_MONH_LIST = [i for i in range(1,13)]

In [3]:
for year in KYS_YEAR_LIST:
    for month in KYS_MONH_LIST:
        file_name = f"history{str(year)}{str(month).zfill(2)}.csv"
        url = f"{KYS_URL}/{file_name}"
        time.sleep(np.random.uniform(1, 5))
        response = requests.get(url)
        if response.status_code == 200:
            print(f"Processing data for year {str(year)}/{str(month).zfill(2)}")
            df = pl.read_csv(source=url, encoding="cp932", skip_rows=1)
            df.write_parquet(os.path.join(KYS_PATH, f"history{str(year)}{str(month).zfill(2)}.parquet"))
            print(f"Saved parquet for year {str(year)}/{str(month).zfill(2)}")
        else:
            print(f"Failed to retrieve data for year {str(year)}/{str(month).zfill(2)}")
            continue

Failed to retrieve data for year 2011/01
Failed to retrieve data for year 2011/02
Failed to retrieve data for year 2011/03
Failed to retrieve data for year 2011/04
Failed to retrieve data for year 2011/05
Failed to retrieve data for year 2011/06
Failed to retrieve data for year 2011/07
Failed to retrieve data for year 2011/08
Failed to retrieve data for year 2011/09
Failed to retrieve data for year 2011/10
Processing data for year 2011/11
Saved parquet for year 2011/11
Processing data for year 2011/12
Saved parquet for year 2011/12
Processing data for year 2012/01
Saved parquet for year 2012/01
Processing data for year 2012/02
Saved parquet for year 2012/02
Processing data for year 2012/03
Saved parquet for year 2012/03
Processing data for year 2012/04
Saved parquet for year 2012/04
Processing data for year 2012/05
Saved parquet for year 2012/05
Processing data for year 2012/06
Saved parquet for year 2012/06
Processing data for year 2012/07
Saved parquet for year 2012/07
Processing dat

### okinawa・hokuriku

In [2]:
start = pl.date(year=2016,month=4,day=1)
end   = pl.date(year=2025,month=12,day=13)

DATE_RANGE = pl.DataFrame({"date": pl.select(pl.date_range(start=start, end=end, interval="1d").alias("date"))}).to_series().to_list()

In [3]:
OKW_PATH = r"/workspace/src/stg/data_lake/electric_forecast/okinawa"
HKR_PATH = r"/workspace/src/stg/data_lake/electric_forecast/hokuriku"
OKW_URL = "https://www.okiden.co.jp/denki2/"
HKR_URL = "https://www.rikuden.co.jp/nw/denki-yoho/csv/juyo_05_20200401.csv"

In [13]:
for t_date in DATE_RANGE:
    date_str = t_date.strftime("%Y%m%d")
    okinawa_url = f"{OKW_URL}/juyo_10_{date_str}.csv"
    time.sleep(np.random.uniform(1, 5))
    res_okinawa = requests.get(okinawa_url)
    print(f"Okinawa : {date_str} SAVE")
    with open(os.path.join(OKW_PATH, f"juyo_10_{date_str}.csv"), 'wb') as f:
        f.write(res_okinawa.content)
    if t_date >= date(2020,4,1):
        hokuriku_url = f"{HKR_URL.replace('20200401', date_str)}"
        time.sleep(np.random.uniform(1, 5))
        res_hokuriku = requests.get(hokuriku_url)
        print(f"Hokuriku : {date_str} SAVE")
        with open(os.path.join(HKR_PATH, f"juyo_05_{date_str}.csv"), 'wb') as f:
            f.write(res_okinawa.content)
    else:
        continue

Okinawa : 20160401 SAVE
Okinawa : 20160402 SAVE
Okinawa : 20160403 SAVE
Okinawa : 20160404 SAVE
Okinawa : 20160405 SAVE
Okinawa : 20160406 SAVE
Okinawa : 20160407 SAVE
Okinawa : 20160408 SAVE
Okinawa : 20160409 SAVE
Okinawa : 20160410 SAVE
Okinawa : 20160411 SAVE
Okinawa : 20160412 SAVE
Okinawa : 20160413 SAVE
Okinawa : 20160414 SAVE
Okinawa : 20160415 SAVE
Okinawa : 20160416 SAVE
Okinawa : 20160417 SAVE
Okinawa : 20160418 SAVE
Okinawa : 20160419 SAVE
Okinawa : 20160420 SAVE
Okinawa : 20160421 SAVE
Okinawa : 20160422 SAVE
Okinawa : 20160423 SAVE
Okinawa : 20160424 SAVE
Okinawa : 20160425 SAVE
Okinawa : 20160426 SAVE
Okinawa : 20160427 SAVE
Okinawa : 20160428 SAVE
Okinawa : 20160429 SAVE
Okinawa : 20160430 SAVE
Okinawa : 20160501 SAVE
Okinawa : 20160502 SAVE
Okinawa : 20160503 SAVE
Okinawa : 20160504 SAVE
Okinawa : 20160505 SAVE
Okinawa : 20160506 SAVE
Okinawa : 20160507 SAVE
Okinawa : 20160508 SAVE
Okinawa : 20160509 SAVE
Okinawa : 20160510 SAVE
Okinawa : 20160511 SAVE
Okinawa : 201605